#property decorator

getter(@property)

In [ ]:
class Student:
    def __init__(self, name, age, grade):
        self.name = name
        self.age = age
        self._grade = grade

    @property
    def get_grade(self):
        return self._grade


In [ ]:
obj = Student("nelson",23,90)
obj.get_grade   #like an attributr ,No parentheses needed

90

In [ ]:
obj.get_grade=70

AttributeError: property 'get_grade' of 'Student' object has no setter

#setter(@age.setter)<br>
a setter is called when you assign a value

<!-- setter(@age.setter)<br>
a setter is called when you assign a value -->

In [ ]:
class Student:
    def __init__(self, name, age, grade):
        self.name = name
        self.age = age
        self._grade = grade

    @property
    def get_grade(self):
        return self._grade

    @get_grade.setter
    def get_grade(self, value):
        if not isinstance(value, int):
            raise TypeError("Grade must be an integer")
        if not 0 <= value <= 100:
            raise ValueError("Grade must be between 0 and 100")
        self._grade = value

In [ ]:
obj = Student("nelson",23,90)
obj.get_grade

90

In [ ]:
obj.get_grade=50

In [ ]:
obj.get_grade

50

#deleter

In [ ]:
class Student:
    def __init__(self, name, age, grade):
        self.name = name
        self.age = age
        self._grade = grade

    @property
    def get_grade(self):
        return self._grade

    @get_grade.setter
    def get_grade(self, value):
        if not isinstance(value, int):
            raise TypeError("Grade must be an integer")
        if not 0 <= value <= 100:
            raise ValueError("Grade must be between 0 and 100")
        self._grade = value

    @get_grade.deleter
    def get_grade(self):
        del self._grade

In [ ]:
obj = Student("nelson",23,90)
obj.get_grade

90

In [ ]:
#Deleter : deleter is called when you used del to delete an attribute
#remove sensative data
del obj.get_grade

In [ ]:
obj.get_grade

AttributeError: 'Student' object has no attribute '_grade'

#Regular Expression

In [ ]:
import re

text = "nelson@gmail.com"

result = re.search(r"@", text)

if result:
    print("@ found")
else:
    print("@ not found")


@ found


checking number is valid or not

In [ ]:
import re

# Regex patterns for different Nepali phone number formats

# Mobile numbers (10 digits, starting with 96/97/98)
MOBILE_RE = re.compile(r'^(?:\+977[- ]?)?(9[678]\d{8})$')

# Mobile + landline (with 977 or 0 prefix)
FULL_PHONE_RE = re.compile(
    r'^(?:\+?977[- ]?)?(?:'
    r'9[678]\d{8}'                # mobile
    r'|(?:0\d{1,2}[- ]?\d{6,7})'  # landline e.g. 01-4xxxxxx, 021-5xxxxx
    r')$'
)

# Any 10-digit number starting with 9 (loose mobile match)
LOOSE_MOBILE_RE = re.compile(r'^(?:\+977)?(?:-)?\d{10}$')


def is_valid_nepali_mobile(number):
    """Validate a Nepali mobile number using regex."""
    return bool(MOBILE_RE.match(number.strip()))


def is_valid_nepali_phone(number):
    """Validate any Nepali phone number (mobile or landline)."""
    return bool(FULL_PHONE_RE.match(number.strip()))


def get_phone_type(number):
    """Return the type of Nepali phone number, or None if invalid."""
    number = number.strip()
    if MOBILE_RE.match(number):
        return 'mobile'
    if FULL_PHONE_RE.match(number):
        return 'landline'
    return None


# Example usage
if __name__ == '__main__':
    test_numbers = [
        '9841234567',        # valid mobile
        '+977-9841234567',   # valid mobile with country code
        '+977 9723456789',   # valid mobile (97 prefix)
        '984123456',         # too short
        '1234567890',        # doesn't start with 9[678]
        '98412345678',       # too long
        '+977 01-4412345',   # valid Kathmandu landline
        '021-5234567',       # valid Biratnagar landline
        '01-4412345',        # valid Kathmandu landline
        '',                  # empty
    ]

    print(f"{'Number':<22} {'Mobile':<8} {'Phone':<8} {'Type'}")
    print('-' * 50)
    for num in test_numbers:
        mobile = is_valid_nepali_mobile(num)
        phone = is_valid_nepali_phone(num)
        ptype = get_phone_type(num) or '-'
        print(f"{num:<22} {str(mobile):<8} {str(phone):<8} {ptype}")

Number                 Mobile   Phone    Type
--------------------------------------------------
9841234567             True     True     mobile
+977-9841234567        True     True     mobile
+977 9723456789        True     True     mobile
984123456              False    False    -
1234567890             False    False    -
98412345678            False    False    -
+977 01-4412345        False    True     landline
021-5234567            False    True     landline
01-4412345             False    True     landline
                       False    False    -


#webscraping

In [ ]:
import re
import requests
from bs4 import BeautifulSoup  # install bs4


def scrape_and_clean(url: str) -> str:
    """
    Scrape webpage and return cleaned text.
    """

    # Request webpage
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0.0.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    # Parse HTML
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove unwanted tags
    for tag in soup([
        "script",
        "style",
        "noscript",
        "header",
        "footer",
        "nav",
        "iframe",
        "svg",
        "img"
    ]):
        tag.decompose()

    # Extract text
    text = soup.get_text(separator=" ")

    # ---------------------
    # REGEX CLEANING
    # ---------------------

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove email addresses abc@gmail.com
    text = re.sub(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        ' ',
        text
    )

    # Remove phone numbers
    text = re.sub(
        r'\+?\d[\d\s\-\(\)]{7,}\d',
        ' ',
        text
    )

    # Remove non-alphanumeric characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # Remove standalone numbers (optional)
    text = re.sub(r'\b\d+\b', ' ', text)

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Strip leading/trailing spaces
    text = text.strip()

    return text


if __name__ == "__main__":
    url = "https://kathmandupost.com/national/2026/08/05/reporter-s-diary-driving-into-a-divided-landscape"

    cleaned_text = scrape_and_clean(url)

    print(cleaned_text[:5000])  # preview first 5000 chars

Reporter s Diary Driving into a divided landscape National Politics Valley Opinion Money Sports Culture Lifestyle National Madhesh Province Lumbini Province Bagmati Province National Security Koshi Province Gandaki Province Karnali Province Sudurpaschim Province Politics Valley Kathmandu Lalitpur Bhaktapur Opinion Letters Columns As it is Editorial Cartoon Money Sports Cricket Football International Sports Culture Lifestyle Arts Brunch with the Post Movies Life Style Theater Entertainment Books Fashion Health Food Recipes Travel Investigations Climate Environment World Science Technology Interviews Visual Stories Crosswords Sudoku Horoscope Forex Corrections Letters to the Editor Today s ePaper What s News Balendra Shah government Nepal s intelligence failure Nepal s EV boom slows AI generated covers PM Shah s meeting with ambassadors National Reporter s Diary Driving into a divided landscape A journey to report on deadly communal violence became an account of fear uncertainty and the 

#search for every headline for the webpage

In [ ]:
import requests
from bs4 import BeautifulSoup


def get_headlines(url):
    """Fetch headlines/titles from a given URL."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    try:
        # 1. Fetch the page
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # 2. Parse HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # 3. Collect headlines from common tags
        headlines = []
        for tag in ['h1', 'h2', 'h3']:
            for element in soup.find_all(tag):
                text = element.get_text(strip=True)
                if text:  # skip empty ones
                    headlines.append(text)

        return headlines

    except requests.exceptions.RequestException as e:
        print(f"Error fetching the page: {e}")
        return []
    except Exception as e:
        print(f"Unexpected error: {e}")
        return []


def save_headlines(url, filename="headlines.txt"):
    """Save headlines to a text file."""
    headlines = get_headlines(url)

    if not headlines:
        print("No headlines found.")
        return

    with open(filename, 'w', encoding='utf-8') as f:
        for i, headline in enumerate(headlines, 1):
            f.write(f"{i}. {headline}\n")

    print(f"Saved {len(headlines)} headlines to {filename}")


# Example usage
if __name__ == '__main__':
    url = input("Enter URL: ").strip()

    lines = get_headlines(url)
    print(f"\nFound {len(lines)} headlines:\n")
    for i, line in enumerate(lines, 1):
        print(f"{i}. {line}")

    # Optionally save to file
    save_headlines(url)

Enter URL: https://en.wikipedia.org/wiki/Spider-Man

Found 26 headlines:

1. Spider-Man
2. Contents
3. Publication history
4. Fictional character biography
5. Personality and themes
6. Powers, skills, and equipment
7. Supporting cast
8. Reception and legacy
9. In other media
10. See also
11. Notes
12. References
13. Bibliography
14. External links
15. Creation and development
16. 1960s
17. 1970s
18. 1980s
19. 1990s
20. 2000s
21. 2010s
22. Enemies
23. Romantic interests
24. Children
25. Alternate versions of Spider-Man
26. Real-life comparisons
Saved 26 headlines to headlines.txt


In [ ]:
#headline generator
import requests
from bs4 import BeautifulSoup
import re
from collections import Counter


def get_page_text(url):
    """Fetch a page and return clean text content."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove unwanted elements that contain repetitive text
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()

        # Extract text from meaningful tags
        text_parts = []
        for tag in ['p', 'title', 'meta']:
            if tag == 'meta':
                meta = soup.find('meta', attrs={'name': 'description'})
                if meta and meta.get('content'):
                    text_parts.append(meta['content'])
            else:
                for el in soup.find_all(tag):
                    t = el.get_text(strip=True)
                    if t:
                        text_parts.append(t)

        return ' '.join(text_parts)

    except Exception as e:
        print(f"Error: {e}")
        return ''


def extract_candidate_sentences(text):
    """Split text into meaningful sentences."""
    # Split on sentence-ending punctuation
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Filter out too-short or too-long sentences
    candidates = [s.strip() for s in sentences if 15 < len(s.strip()) < 200]
    return candidates


def generate_headlines(url, count=5):
    """Generate headlines from webpage content when headings are missing."""
    text = get_page_text(url)
    if not text:
        return ['No content found on the page.']

    sentences = extract_candidate_sentences(text)

    # Score sentences by keyword frequency (simple extractive approach)
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    stopwords = {'the', 'and', 'that', 'this', 'with', 'from', 'have', 'were',
                 'their', 'there', 'these', 'will', 'would', 'they', 'been'
                 }  # common English stopwords
    word_freq = Counter(w for w in words if w not in stopwords)

    scored = []
    for s in sentences:
        s_words = re.findall(r'\b[a-zA-Z]{4,}\b', s.lower())
        if not s_words:
            continue
        score = sum(word_freq.get(w, 0) for w in s_words) / len(s_words)
        scored.append((score, s))

    # Sort by score descending, take top N
    scored.sort(reverse=True, key=lambda x: x[0])
    top = [s for _, s in scored[:count]]

    # Title case the start of each generated headline
    headlines = []
    for s in top:
        s = s.rstrip('.!?')  # remove trailing punctuation
        headlines.append(s)

    return headlines if headlines else ['Could not generate headlines.']


# Main usage
if __name__ == '__main__':
    url = input("Enter URL: ").strip()

    # Try scraping headings first
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        real_headlines = []
        for tag in ['h1', 'h2', 'h3']:
            for el in soup.find_all(tag):
                t = el.get_text(strip=True)
                if t:
                    real_headlines.append(t)
    except Exception as e:
        real_headlines = []
        print(f"Could not fetch page: {e}")

    if real_headlines:
        print("\n-- Real headlines found --")
        for i, h in enumerate(real_headlines, 1):
            print(f"{i}. {h}")
    else:
        print("\n-- No headings found. Generating headlines from content... --")
        gen = generate_headlines(url)
        for i, h in enumerate(gen, 1):
            print(f"{i}. {h}")

Enter URL: https://kathmandupost.com/national/2026/08/05/reporter-s-diary-driving-into-a-divided-landscape

-- Real headlines found --
1. Without Fear or FavourUNWIND IN STYLE
2. Reporter’s Diary: Driving into a divided landscape


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from collections import Counter


def get_page_text(url):
    """Fetch a page and return clean text content."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove unwanted elements that contain repetitive text
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()

        # Extract text from meaningful tags
        text_parts = []
        for tag in ['p', 'title', 'meta']:
            if tag == 'meta':
                meta = soup.find('meta', attrs={'name': 'description'})
                if meta and meta.get('content'):
                    text_parts.append(meta['content'])
            else:
                for el in soup.find_all(tag):
                    t = el.get_text(strip=True)
                    if t:
                        text_parts.append(t)

        return ' '.join(text_parts)

    except Exception as e:
        print(f"Error: {e}")
        return ''


def extract_candidate_sentences(text):
    """Split text into meaningful sentences."""
    # Split on sentence-ending punctuation
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Filter out too-short or too-long sentences
    candidates = [s.strip() for s in sentences if 15 < len(s.strip()) < 250]
    return candidates


def score_sentences(sentences, text):
    """Score sentences by keyword frequency."""
    stopwords = {'the', 'and', 'that', 'this', 'with', 'from', 'have', 'were',
                 'their', 'there', 'these', 'will', 'would', 'they', 'been',
                 'would', 'could', 'should', 'more', 'than', 'into', 'also'}
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    word_freq = Counter(w for w in words if w not in stopwords)

    scored = []
    for s in sentences:
        s_words = re.findall(r'\b[a-zA-Z]{4,}\b', s.lower())
        if not s_words:
            continue
        score = sum(word_freq.get(w, 0) for w in s_words) / len(s_words)
        scored.append((score, s))

    scored.sort(reverse=True, key=lambda x: x[0])
    return scored


def make_variants(headline):
    """Generate alternative phrasings of a single headline."""
    clean = headline.rstrip('.!?')
    variants = set()  # use a set to avoid duplicates

    # 1. Question form
    variants.add(f"What if {clean[0].lower()}{clean[1:]}?")

    # 2. 'Why' / 'How' prefix
    variants.add(f"Why {clean[0].lower()}{clean[1:]}")
    variants.add(f"How {clean[0].lower()}{clean[1:]}")

    # 3. Add 'Breaking' prefix
    variants.add(f"Breaking: {clean}")

    # 4. Add "In today's news" prefix
    variants.add(f"In today's news: {clean}")

    # 5. Emphasized / exclamation version
    variants.add(f"{clean}!")

    # 6. 'Report:' prefix
    variants.add(f"Report: {clean}")

    return list(variants)


def generate_headlines_with_alternatives(url, count=5, alts_per=3):
    """
    Generate main headlines + alternative versions.
    Returns a list of (main, [alternatives]) tuples.
    """
    text = get_page_text(url)
    if not text:
        return [('No content found on the page.', [])]

    sentences = extract_candidate_sentences(text)
    scored = score_sentences(sentences, text)
    top = [s for _, s in scored[:count]]

    if not top:
        return [('Could not generate headlines.', [])]

    result = []
    for main in top:
        main = main.rstrip('.!?')
        variants = make_variants(main)
        # Cap the number of alternatives per headline
        result.append((main, variants[:alts_per]))

    return result


# Main usage
if __name__ == '__main__':
    url = input("Enter URL: ").strip()

    # Try scraping headings first
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        real_headlines = []
        for tag in ['h1', 'h2', 'h3']:
            for el in soup.find_all(tag):
                t = el.get_text(strip=True)
                if t:
                    real_headlines.append(t)
    except Exception:
        real_headlines = []

    if real_headlines:
        print("\n-- Real headlines found --")
        for i, h in enumerate(real_headlines, 1):
            print(f"{i}. {h}")
            # Also show alternatives for real headlines
            for j, alt in enumerate(make_variants(h)[:3], 1):
                print(f"   Alt {j}: {alt}")
    else:
        print("\n-- No headings found. Generating headlines from content... --")
        generated = generate_headlines_with_alternatives(url)
        for i, (main, alts) in enumerate(generated, 1):
            print(f"{i}. {main}")
            for j, alt in enumerate(alts, 1):
                print(f"   Alt {j}: {alt}")

Enter URL: https://kathmandupost.com/national/2026/08/05/reporter-s-diary-driving-into-a-divided-landscape

-- Real headlines found --
1. Without Fear or FavourUNWIND IN STYLE
   Alt 1: Without Fear or FavourUNWIND IN STYLE!
   Alt 2: Report: Without Fear or FavourUNWIND IN STYLE
   Alt 3: In today's news: Without Fear or FavourUNWIND IN STYLE
2. Reporter’s Diary: Driving into a divided landscape
   Alt 1: Why reporter’s Diary: Driving into a divided landscape
   Alt 2: Breaking: Reporter’s Diary: Driving into a divided landscape
   Alt 3: Reporter’s Diary: Driving into a divided landscape!


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def get_title_and_description(url: str) -> dict:
    """Fetches the title and meta description from a given URL."""
    headers = {
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36' +
                       ' (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36')
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)

        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract title
        title = soup.find('title')
        title_text = title.get_text(strip=True) if title else 'No Title Found'

        # Extract meta description
        description = soup.find('meta', attrs={'name': 'description'})
        description_text = description['content'].strip() if description and 'content' in description.attrs else 'No Description Found'

        return {
            'URL': url,
            'Title': title_text,
            'Description': description_text
        }

    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return {
            'URL': url,
            'Title': 'Error: Could not access page',
            'Description': str(e)
        }
    except Exception as e:
        print(f"An unexpected error occurred for {url}: {e}")
        return {
            'URL': url,
            'Title': 'Error: Unexpected issue',
            'Description': str(e)
        }

def process_urls_and_save_to_csv(urls: list, output_filename: str = 'news_articles.csv'):
    """Processes a list of URLs to extract title/description and saves to CSV."""
    if not urls:
        print("No URLs provided to process.")
        return

    all_data = []
    for i, url in enumerate(urls):
        print(f"Processing URL {i+1}/{len(urls)}: {url}")
        data = get_title_and_description(url)
        all_data.append(data)

    df = pd.DataFrame(all_data)
    df.to_csv(output_filename, index=False)
    print(f"Successfully saved {len(df)} articles to {output_filename}")


# Example Usage:
if __name__ == '__main__':
    # Please replace this with your list of up to 100 news article URLs.
    # For example:
    # news_urls = [
    #    "https://example.com/news/article1",
    #    "https://example.com/news/article2",
    #    ...
    # ]
    news_urls = [
        "https://kathmandupost.com/national/2026/08/05/reporter-s-diary-driving-into-a-divided-landscape",
        "https://www.theguardian.com/world/2024/may/17/ukraine-war-russia-kharkiv-shelling-front-lines",
        "https://www.bbc.com/news/world-asia-69022630",
        "https://www.nytimes.com/interactive/2024/05/17/us/ai-generated-images-election.html"
        # Add up to 96 more URLs here
    ]

    # Limit to a maximum of 100 URLs as requested
    news_urls = news_urls[:100]

    process_urls_and_save_to_csv(news_urls, 'news_data.csv')


Processing URL 1/4: https://kathmandupost.com/national/2026/08/05/reporter-s-diary-driving-into-a-divided-landscape
Processing URL 2/4: https://www.theguardian.com/world/2024/may/17/ukraine-war-russia-kharkiv-shelling-front-lines
Error accessing https://www.theguardian.com/world/2024/may/17/ukraine-war-russia-kharkiv-shelling-front-lines: 404 Client Error: Not Found for url: https://www.theguardian.com/world/2024/may/17/ukraine-war-russia-kharkiv-shelling-front-lines
Processing URL 3/4: https://www.bbc.com/news/world-asia-69022630
Error accessing https://www.bbc.com/news/world-asia-69022630: 404 Client Error: Not Found for url: https://www.bbc.com/news/world-asia-69022630
Processing URL 4/4: https://www.nytimes.com/interactive/2024/05/17/us/ai-generated-images-election.html
Error accessing https://www.nytimes.com/interactive/2024/05/17/us/ai-generated-images-election.html: 403 Client Error: Forbidden for url: https://www.nytimes.com/interactive/2024/05/17/us/ai-generated-images-electio

In [ ]:
import random
import string

def generate_random_news_urls(count: int = 100) -> list:
    """Generates a list of random, plausible-looking news article URLs."""
    base_domains = [
        "www.example.com",
        "news.test.org",
        "www.fakenews.info",
        "journal.co",
        "dailystories.net"
    ]
    categories = [
        "politics",
        "technology",
        "sports",
        "entertainment",
        "world",
        "business"
    ]
    years = [str(y) for y in range(2020, 2025)]
    months = [f"{m:02d}" for m in range(1, 13)]
    days = [f"{d:02d}" for d in range(1, 29)]

    random_urls = []
    for _ in range(count):
        domain = random.choice(base_domains)
        category = random.choice(categories)
        year = random.choice(years)
        month = random.choice(months)
        day = random.choice(days)

        # Generate a random article slug
        slug_length = random.randint(5, 15)
        slug = ''.join(random.choice(string.ascii_lowercase + string.digits) for _ in range(slug_length))

        url = f"https://{domain}/{category}/{year}/{month}/{day}/{slug}-article"
        random_urls.append(url)

    return random_urls

# Generate 100 random URLs
if __name__ == '__main__':
    generated_urls = generate_random_news_urls(100)
    print(f"Generated {len(generated_urls)} random URLs. Here are the first 5:")
    for i, url in enumerate(generated_urls[:5]):
        print(f"{i+1}. {url}")

    # You can now use this list with the previous function:
    # process_urls_and_save_to_csv(generated_urls, 'random_news_data.csv')


Generated 100 random URLs. Here are the first 5:
1. https://journal.co/entertainment/2021/05/04/1wods005-article
2. https://www.example.com/entertainment/2024/01/13/af6225g-article
3. https://dailystories.net/sports/2023/01/27/u63hkt1ppuwq-article
4. https://dailystories.net/world/2022/05/06/nkbuke7bq-article
5. https://www.fakenews.info/entertainment/2024/09/08/jja5r5o56cp1-article


In [ ]:
# Use the generated URLs with the scraping function
process_urls_and_save_to_csv(generated_urls, 'random_news_data.csv')
print("Processing of random URLs complete. Check 'random_news_data.csv'.")

Processing URL 1/100: https://journal.co/entertainment/2021/05/04/1wods005-article
Processing URL 2/100: https://www.example.com/entertainment/2024/01/13/af6225g-article
Error accessing https://www.example.com/entertainment/2024/01/13/af6225g-article: 404 Client Error: Not Found for url: https://www.example.com/entertainment/2024/01/13/af6225g-article
Processing URL 3/100: https://dailystories.net/sports/2023/01/27/u63hkt1ppuwq-article
Processing URL 4/100: https://dailystories.net/world/2022/05/06/nkbuke7bq-article
Processing URL 5/100: https://www.fakenews.info/entertainment/2024/09/08/jja5r5o56cp1-article
Processing URL 6/100: https://www.example.com/politics/2023/06/15/q5nby2-article
Error accessing https://www.example.com/politics/2023/06/15/q5nby2-article: 404 Client Error: Not Found for url: https://www.example.com/politics/2023/06/15/q5nby2-article
Processing URL 7/100: https://journal.co/technology/2021/03/03/cqojgv-article
Processing URL 8/100: https://journal.co/business/202